In [ ]:
import yaml
import os

config_path = os.path.join(os.getcwd(), "config.yml")

with open(config_path) as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

In [ ]:
import pymongo

client = pymongo.MongoClient(config["client"])
db = client[config["db"]]
coll = db[config["col"]]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from utils import apply_filter, sliding_window_pd

#extract features

def process_sensor_data(df, trim_size=100, z_threshold=3):
    df_trimmed = df.iloc[:-trim_size].copy()
    for col_name in df_trimmed.columns:
        mean_val = df_trimmed[col_name].mean()
        std_val = df_trimmed[col_name].std()
        z_scores = (df_trimmed[col_name] - mean_val) / std_val
        df_trimmed.loc[np.abs(z_scores) > z_threshold, col_name] = np.nan
        df_trimmed[col_name] = df_trimmed[col_name].interpolate(method='linear')
        df_trimmed[col_name] = df_trimmed[col_name].ffill().bfill()
    return df_trimmed

def extract_features(window_df):
    features = []
    for col in window_df.columns:
        features.append(window_df[col].mean())
        features.append(window_df[col].std())
        features.append(window_df[col].max())
        features.append(window_df[col].min())
        features.append(window_df[col].var())
    return features

X_global_list = []
y_labels_list = []
user_ids_list = []

order = config["filter"]["order"]
wn = config["filter"]["wn"]
window_size = config["sliding_window"]["ws"]
overlap = config["sliding_window"]["overlap"]

all_documents = coll.find({})

for doc in all_documents:
    gesture = doc.get("gesture_id")
    user = doc.get("user")
    df = pd.DataFrame(doc["data"])
    
    df_clean = process_sensor_data(df)
    df_filtered = df_clean.apply(apply_filter, args=(order, wn, "lowpass"))
    
    windows = sliding_window_pd(df_filtered, ws=window_size, overlap=overlap, print_stats=False)
    
    for window in windows:
       
        window_features = extract_features(window)
        
        X_global_list.append(window_features)
        y_labels_list.append(gesture)
        user_ids_list.append(user)

X_global = np.array(X_global_list)
y_labels = np.array(y_labels_list)
user_ids = np.array(user_ids_list)

train_mask = user_ids == "01"
test_mask = user_ids == "02"

X_train_raw = X_global[train_mask]
y_train_raw = y_labels[train_mask]
X_test_raw = X_global[test_mask]
y_test_raw = y_labels[test_mask]

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print(f"Συνολική Ακρίβεια (Accuracy): {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=le.classes_, cmap='Blues')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()